# Transformers and Tap Changers

----

The 61970 Wires package defines several classes for representing the different components of transformers. The first is PowerTransformer, which represents the electrical network representation of a transformer for both balanced and unbalanced circuits. The second is TransformerTank, which refers to the assembly of two or more windings placed inside a tank and can be used to model single-phase and three-phase transformers. TransformerEnd is the conducting connection point of a transformer and corresponds to the terminal of a particular winding.  As with all ConductingEquipment, the PowerTransformer is connected through a set of Terminal objects, with the particular winding indicated by the TransformerEnd.endNumber attribute. The highest voltage winding should have an endNumber of 1. The endNumber does not need to match the ACDCTerminal.sequenceNumber attribute of the Terminal to which the transformer is connected. 

The BaseVoltage and Terminal are associated with the TransformerEnd of each winding, rather than with the PowerTransformer itself. Figure 16 shows the associations between the different classes used to specify transformer parameters and windings.

PowerTransformer objects may be modeled with or without specifying TransformerTank objects. In both cases the PowerTransformer.vectorGroup attribute for protective relaying should be specified according to IEC transformer standards (e.g., Dy1 for many substation transformers). 


The case without specifying TransformerTank objects is most suitable for balanced three-phase transformers that will not reference any reusable asset catalog data. This approach is typically used for transmission system modeling, where each transformer is unique. Each winding will have a PowerTransformerEnd that associates to both a Terminal and a BaseVoltage, and the parent PowerTransformer. The impedance and admittance parameters are defined by reverse-associated TransformerMeshImpedance between each pair of windings, and a reverse-associated TransformerCoreAdmittance for one winding. The units for these are ohms and siemens based on the winding voltage, rather than per-unit. WindingConnection is similar to PhaseShuntConnectionKind, adding Z and Zn for zig-zag connections and A for autotranformers. TransformerStarImpedance is used for conversion of three-winding transformers to separate two-winding equivalents, which is common practice in numerous power flow solvers.

If the transformer is unbalanced in any way, then TransformerTankEnd is used instead of PowerTransformerEnd, and then one or more TransformerTank objects may be used in the parent PowerTransformer. Some of the use cases are 1) center-tapped secondary, 2) open-delta and 3) EHV transformer banks. Tank-level modeling is also required if using catalog data to specify physical equipment ratings, etc. through the AssetInfo package. 



In [1]:
from cimgraph import utils
from mermaid import Mermaid
import cimgraph.data_profile.cim17v40 as cim

In [9]:
diagram_text = utils.get_mermaid([cim.TransformerEnd, cim.Terminal, cim.ConductingEquipment, cim.PowerTransformer, cim.BaseVoltage, cim.PowerTransformerEnd, cim.TransformerTank, cim.TransformerTankEnd, cim.TransformerStarImpedance, cim.TransformerMeshImpedance,cim.TransformerCoreAdmittance, cim.PhaseCode, cim.WindingConnection])
Mermaid(diagram_text)

Many distribution software packages use the concept of catalog data, aka library data, especially for lines and transformers. This concept is implemented in CIM through the ability to define a single set of class definitions using the IEC 61968 AssetInfo package to save a large amount of space when defining customer secondary transformers (which typically comprise hundreds or thousands of identical poletop and pad-mounted installations). A particular transformer design and rating is then defined by creating PowerTransformerInfo and TransformerTankInfo library objects that are associated with each type of specification objects. 

The rated voltage, rated current, and resistance of each winding are defined as attributes of a TransformerEndInfo object that is created for each transformer winding. It is important that the TransformerEndInfo.endNumber of the physical asset match the TransformerEnd.endNumber of its representation in the electrical circuit. The shunt admittances are defined by NoLoadTest on a winding / end, with usually just one such test. The impedances are defined by a set of attributes of ShortCircuitTest; one winding / end will be energized, and one or more of the others will be grounded in these tests. The complete list of asset properties is summarized in Figure 17. Note that these classes are associated with TransformerTankInfo (not PowerTransformerInfo) because transformer testing is done on tanks.

In [13]:
diagram_text = utils.get_mermaid([cim.TransformerTankInfo, cim.PowerTransformerInfo, cim.TransformerEndInfo, cim.OpenCircuitTest, cim.ShortCircuitTest, cim.NoLoadTest, cim.TransformerTest, cim.WindingConnection])
Mermaid(diagram_text)

The TapChanger class is used to model both phase-shifting transformers and voltage regulators through the PhaseTapChanger and RatioTapChanger classes, which are associated with the particular TransformerEnd. The highest, lowest, and neutral tap positions available are specified as positive integers such that a TapChanger with 16 tap positions would have attributes of lowStep set to 0, neutralStep set to 8, and highStep set to 16. If a particular application uses a range of -8 to 8 for the same transformer tap range, it is the responsibility of application to convert the tap ranges to the format used internally. 

If a voltage regulator uses line drop compensation, then those parameters will be defined as attributes of the TapChangerControl class, which inherits from RegulatingControl. RegulatingControl is a higher-level class that is used to specify the control mode and setpoints for numerous types of RegulatingCondEq, such capacitors, reactors, SVC, and generator automatic voltage regulation controls. Whether a particular device is regulating voltage, activePower, reactivePower, etc. is specified by the RegulatingControl:mode attribute. Other control settings, such as targetValue, targetDeadband, etc.  are also attributes of RegulatingControl, as shown in Figure 18 below.

In summary, a single-phase line voltage regulator modeled in CIM includes a PowerTransformer, a TransformerTank, a TransformerTankEnd, a RatioTapChanger, and a TapChangerControl. The CT and PT parameters of a voltage regulator can only be described via the AssetInfo mechanism, described below. The RegulationControl.mode must be voltage. Older CIM versions used the tculControlMode attribute, which is now deprecated. 

The AssetInfo package in the 61968 package defines the TapChangerInfo class with a set of attributes ctRating, ctRatio, and ptRatio needed for line drop compensator settings in voltage regulators. Catalog data is a one-to-many relationship. In this case, many TapChangers can share the same TapChangerInfo data, which saves space and provides consistency. Older versions of CIM had many-to-many catalog relationships, but now only one AssetDataSheet may be associated per Equipment.


In [19]:
diagram_text = utils.get_mermaid([cim.PowerTransformerEnd, cim.TransformerTankEnd, cim.TransformerEnd, cim.RatioTapChanger,cim.TapChanger, cim.TapChangerControl, cim.RegulatingControl, cim.RegulatingControlModeKind, cim.PhaseCode])
Mermaid(diagram_text)

Some examples are discussed below.

In [9]:
from cimgraph.models import FeederModel
from cimgraph.databases import ConnectionParameters, XMLFile
import cimgraph.data_profile.cimhub_2023 as cim
import json
from uuid import UUID
cim_profile = 'cimhub_2023'

In [2]:
params = ConnectionParameters(filename='../sample_models/ieee13.xml',
                              cim_profile='cimhub_2023',
                              iec61970_301=8) # file path
file = XMLFile(params) # file read connection
network = FeederModel(container=cim.Feeder(),connection=file) # create feeder model

Example 1: How many triplex secondary transformers are in the model?

In [3]:
# Split-phase transformers will have 1 TransformerTank and 3 Terminal and None PowerTransformerEnd
# Graph traversal path is PowerTransformer -> List[Terminal]

results = []

# Loop through all PowerTransformer in the network
for power_transformer in network.graph[cim.PowerTransformer].values():
    # Count the number of TransformerTank objects. We expect one
    num_tanks = len(power_transformer.TransformerTanks)
    # Count the number of Terminal objects. We expect three
    num_terminal = len(power_transformer.Terminals)
    # Count the number of PowerTransformerEnd. We expect None / zero
    num_3_ph_windings = len(power_transformer.PowerTransformerEnd)

    # If it has one TransformerTank, three Terminal, and zero windings, it is a split-phase secondary transformer
    if num_tanks == 1 and num_terminal == 3 and num_3_ph_windings == 0:
        # Add the name of the split-phase service transformer
        results.append(power_transformer.name)
results = set(results)

print(results)

{'tpoletop'}


Example 2: Which split-phase transformers are connected to phase B? Do not include voltage regulators.

In [4]:
# Voltage regulators will have 3 tanks and 6 terminals or 1 tank and 2 terminals
# Split-phase transformers will have 1 tank and 3 terminals
# The high-side winding will be connected to phase A, B, or C
# The low-side winding will be connected to phase s1 and s2

results = []

# Loop through all PowerTransformer in the network
for power_transformer in network.graph[cim.PowerTransformer].values():
    # Count the number of TransformerTank objects. We expect one
    num_tanks = len(power_transformer.TransformerTanks)
    # Count the number of Terminal objects. We expect three
    num_terminal = len(power_transformer.Terminals)

    # If it has one TransformerTank, three Terminal, it is a split-phase secondary transformer
    if num_tanks == 1 and num_terminal == 3:
        # Get the TransformerTank of the single-phase transformer
        transformer_tank = power_transformer.TransformerTanks[0]
        # Loop through all TransformerTankEnd windings of the TransformerTank
        for tank_end in transformer_tank.TransformerTankEnds:
            if 'B' in str(tank_end.orderedPhases):
                results.append(power_transformer.name)

print(results)

['tpoletop']


Example 3: What are the voltage ratings of each winding for the single-phase transformer tank with mRID "17A934C7-1510-481F-BAD7-189058957FF1?

In [11]:

mRID = "17A934C7-1510-481F-BAD7-189058957FF1"
results = []

# Final attribute is cim.PowerTransformerEnd.ratedU. 
# Graph traversal path is PowerTransformer -> List[PowerTransformerEnd]

# Convert the mRID to a UUID object
uuid = UUID(mRID.strip('_').lower())

transformer_tank = network.graph[cim.TransformerTank][uuid]
# Check if the TransformerTank has an TransformerTankInfo asset datasheet
if transformer_tank.TransformerTankInfo is not None:
    tank_info = transformer_tank.TransformerTankInfo
    # Loop through all TransformerEndInfo datasheets for the TransformerTankInfo datasheet
    for end_info in tank_info.TransformerEndInfos:
        winding = dict()
        # The endNumber specifies which end. The high-side is endNumber=1 and low-side is endNumber=2,3
        winding['end_number'] = end_info.endNumber
        # The nominal voltage rating of each winding is given by the TransformerEndInfo.ratedU attribute
        winding['rated_voltage'] = end_info.ratedU
        results.append(winding)

print(results)

[{'end_number': 1, 'rated_voltage': 2400.0}, {'end_number': 2, 'rated_voltage': 120.0}, {'end_number': 3, 'rated_voltage': 120.0}]


Example 4: How many three-winding transformers are in the model?

In [12]:
# Three-winding transformers will have 3 Terminal AND 3 PowerTransformerEnd objects
# Graph path traversal is PowerTransformer -> List[Terminal] and PowerTransformer -> List[PowerTransformerEnd]

results = []

# Loop through all power_transformer objects in the network graph
for power_transformer in network.graph[cim.PowerTransformer].values():
    # Count the number of terminals connected to the transformer
    terminal_count = len(power_transformer.Terminals)
    # Count the number of windings associated with the transformer
    winding_count = len(power_transformer.PowerTransformerEnd)
    # If it has 3 Terminal and 3 PowerTransformerEnd, it is a 3-winding transformer
    if terminal_count == 3 and winding_count == 3:
        results.append(power_transformer.name)

print(results)

['sub3']


Example 5: How many two-winding transformers are in the model?

In [13]:
# Three-winding transformers will have 2 Terminal AND 2 PowerTransformerEnd objects
# Graph path traversal is PowerTransformer -> List[Terminal] and PowerTransformer -> List[PowerTransformerEnd]

results = []

# Loop through all power_transformer objects in the network graph
for power_transformer in network.graph[cim.PowerTransformer].values():
    # Count the number of terminals connected to the transformer
    terminal_count = len(power_transformer.Terminals)
    # Count the number of windings associated with the transformer
    winding_count = len(power_transformer.PowerTransformerEnd)
    # If it has 2 Terminal and 2 PowerTransformerEnd, it is a 2-winding transformer
    if terminal_count == 2 and winding_count == 2:
        results.append(power_transformer.name)
print(results)

['xfm1']


Example 6: What are the names of buses that transformer name 'xfm1' is connected to?

In [14]:
name = 'xfm1'
results = []

# Final attribute is cim.ConnectivityNode.name
# Graph path traversal is PowerTransformer -> List[Terminal] -> ConnectivityNode

# Loop through all power_transformer objects in the network graph
for power_transformer in network.graph[cim.PowerTransformer].values():
    # Check if the name matches
    if power_transformer.name == name:
        # Loop through all terminals connected to the transformer
        for terminal in power_transformer.Terminals:
            # Get the node connected to each terminal
            node = terminal.ConnectivityNode
            results.append(node.name)

print(results)

['xf1', '634']


Example 7: What is the nominal voltage ratings of three-phase transformer with mRID "1E6B5C97-C4E8-4CED-B9A5-6E69F389DA93?

In [15]:
mRID = "1E6B5C97-C4E8-4CED-B9A5-6E69F389DA93"
results = []

# Final attribute is cim.PowerTransformerEnd.ratedU. 
# Graph traversal path is PowerTransformer -> List[PowerTransformerEnd]

# Convert the mRID to a UUID object
uuid = UUID(mRID.strip('_').lower())

# Get the three-phase transformer with the correct uuid
power_transformer = network.graph[cim.PowerTransformer][uuid]

# Loop through all PowerTransformerEnd windings connected
for power_transformer_end in power_transformer.PowerTransformerEnd:
    # Get the rated nominal voltage of each PowerTransformerEnd
    results.append(power_transformer_end.ratedU)

print(results)

[4160.0, 480.0]


Example 8: What is the apparent power rating of each three-phase transformer in the feeder?

In [16]:
# Three-phase transformers will have have an assocation to more than one PowerTransformerEnd defined
# Final attribute is cim.PowerTransformerEnd.ratedS
# Graph path traversal is PowerTransformer -> List[PowerTransformerEnd] 

results = []

# Loop through all PowerTransformer objects in the network graph
for power_transformer in network.graph[cim.PowerTransformer].values():
    # If there are PowerTransformerEnd objects, then it is a 3-ph xfmr
    if power_transformer.PowerTransformerEnd != []:
        # Create output structure for name and ratedS of each winding
        rating = dict()
        rating['xfmr_name'] = power_transformer.name
        rating['VA_rating'] = []
        # Each winding is represented by a PowerTransformerEnd in a list
        for power_transformer_end in power_transformer.PowerTransformerEnd:
            # The apparerent power rating is given by the ratedS attribute
            rating['VA_rating'].append(power_transformer_end.ratedS)
        results.append(rating)
        
print(results)

[{'xfmr_name': 'sub3', 'VA_rating': [5000000.0, 5000000.0, 1000000.0]}, {'xfmr_name': 'xfm1', 'VA_rating': [500000.0, 500000.0]}]


Example 9: How is the three-phase transformer named 'xfm1' connected?

In [17]:
name = 'xfm1'
results = dict()

# Final attribute is cim.ConnectivityNode.name
# Graph path traversal is PowerTransformer -> List[Terminal] -> ConnectivityNode

# Loop through all power_transformer objects in the network graph
for power_transformer in network.graph[cim.PowerTransformer].values():
    # Check if the name matches
    if power_transformer.name == name:
        # the PowerTransformer.vectorGroup describes the winding connnection
        results['vector_group'] = str(power_transformer.vectorGroup)
        for power_transformer_end in power_transformer.PowerTransformerEnd:
            # The high-side winding has an endNumber of 1. This winding has the higher voltage.
            if power_transformer_end.endNumber == 1:
                # Each winding has a connectionKind enumeration
                results['high_side_connection'] = str(power_transformer_end.connectionKind)
            # The low-side winding has an endNumber of 2. This winding has the lower voltage.
            elif power_transformer_end.endNumber == 2:
                # Each winding has a connectionKind enumeration
                results['low_side_connection'] = str(power_transformer_end.connectionKind)

print(results)

{'vector_group': 'Yy', 'high_side_connection': 'WindingConnection.Y', 'low_side_connection': 'WindingConnection.Y'}


Example 10: How many windings does transformer with mRID "1E6B5C97-C4E8-4CED-B9A5-6E69F389DA93" have?

In [18]:
mRID = "1E6B5C97-C4E8-4CED-B9A5-6E69F389DA93"
results = []

# Final attribute is cim.PowerTransformer.PowerTransformerEnd
# Graph traversal path is PowerTransformer -> List[PowerTransformerEnd]

# Convert the mRID to a UUID object
uuid = UUID(mRID.strip('_').lower())

# Get the PowerTransformer with correct uuid
power_transformer = network.graph[cim.PowerTransformer][uuid]
# Count the total number of PowerTransformerEnd connected
winding_count = len(power_transformer.PowerTransformerEnd)
result = winding_count
print(result)

2


Example 11: What is the impedance of the transformer with mRID "1E6B5C97-C4E8-4CED-B9A5-6E69F389DA93"? Express the impedance on the basis of the nodes connected to each winding.

In [19]:
mRID = "1E6B5C97-C4E8-4CED-B9A5-6E69F389DA93"
results = dict()

# Final attribute is cim.PowerTransformerEnd.r, cim.TransformerMeshImpedance.r, cim.TransformerMeshImpedance.x, and cim.TransformerCoreAdmittance.b
# Graph traversal path is PowerTransformer -> List[PowerTransformerEnd] -> Terminal -> ConnectivityNode
# Graph traversal path is PowerTransformer -> List[PowerTransformerEnd] -> List[cim.TransformerMeshImpedance]
# Graph traversal path is PowerTransformer -> List[PowerTransformerEnd] -> cim.TransformerCoreAdmittance.b

# Convert the mRID to a UUID object
uuid = UUID(mRID.strip('_').lower())

power_transformer = network.graph[cim.PowerTransformer][uuid]

# Loop through all PowerTansformerEnd winding objects connected to the transformer
for power_transformer_end in power_transformer.PowerTransformerEnd:
    # The high-side winding has an endNumber of 1. This winding has the higher voltage.
    if power_transformer_end.endNumber == 1:
        # To get the high-side bus, go through the Terminal and then the ConnectivityNode
        results['high_side_bus'] = power_transformer_end.Terminal.ConnectivityNode.name
        # Each winding has an internal resistance
        results['high_side_resistance'] = power_transformer_end.r
        
        # The impendance from the high-side to the low-side is TransformerMeshImpedance
        if power_transformer_end.FromMeshImpedance != []:
            # High-side has a FromMeshImpedance. The low-side has a ToMeshImpedance.
            mesh_impedance = power_transformer_end.FromMeshImpedance[0]
            # Get the resistance and reactance values of the TransformerMeshImpedance
            results['x'] = mesh_impedance.x
            results['r'] = mesh_impedance.r
        
        # Get the conductance and susceptance of the transformer core
        if power_transformer_end.CoreAdmittance is not None:
            # Get the TransformerCoreAdmittance object
            core_admittance = power_transformer_end.CoreAdmittance
            results['b'] = core_admittance.b
            results['g'] = core_admittance.g
            
    # The low-side winding has an endNumber of 2. This winding has the lower voltage.
    elif power_transformer_end.endNumber == 2:
        # To get the high-side bus, go through the Terminal and then the ConnectivityNode
        results['low_side_bus'] = power_transformer_end.Terminal.ConnectivityNode.name
        # Each winding has an internal resistance
        results['low_side_resistance'] = power_transformer_end.r

print(results)

{'high_side_bus': 'xf1', 'high_side_resistance': 0.1903616, 'x': 0.692224, 'r': 0.3807232, 'b': 0.0, 'g': 0.0, 'low_side_bus': '634', 'low_side_resistance': 0.0025344}
